# Case 10 — Cyclic features & stream stability for LSTM-VAE

**Reproduces:** Fig 4.22

Group anomalies, window mode. The thesis found that appending cyclic features to an LSTM-VAE (without reconstructing them) can destabilise its recurrent temporal representation: worse overall F1, but with a distinctive early-stream dip and recovery pattern rather than a uniform decline. The bucketed view below is what shows this — the aggregate F1 in the table above hides it.

Runtime menu -> Change runtime type -> GPU, then run all cells.

This notebook runs on the real ERA5 slice shipped with the repo under `data/era5/` (24 years, centered on the same warmup/test split used at full scale, split exactly 50:50 warmup/test — see `DATA_LICENSE.md`) — no download needed. Numbers will still differ from the thesis's full-scale figures (much shorter warmup/test period, noisier), but the *qualitative* effect described above should still show up.

**Setup:** this is a private repo, so before running you need a GitHub token as a Colab secret — key icon in the left sidebar -> new secret named `GITHUB_TOKEN`, value = a token from [github.com/settings/tokens](https://github.com/settings/tokens) (read-only `repo` access is enough), then toggle "Notebook access" on.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Private repo: add a GitHub token as a Colab secret first --
    # key icon in the left sidebar -> Secrets -> new secret named GITHUB_TOKEN,
    # value = a token from github.com/settings/tokens (read-only "repo" access is enough)
    # -> toggle "Notebook access" on.
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")
    if not os.path.exists("repo"):
        !git clone $clone_url repo
    %cd repo
elif not os.path.exists("run_regression.py"):
    # Already inside a local checkout (e.g. running from notebooks/cases/) --
    # move to the repo root instead of cloning a redundant nested copy.
    %cd ../..

!pip install -q -r requirements.txt


In [ ]:
import os
import yaml

# Edit this directly — works the same locally and on Colab. Every option
# below names an explicit data_source layer file (or None for the base
# anom_types/group.yaml default):
#   "full_split_files" (default) -> full-scale group-anomaly data,
#                      pre-split into a warmup-only file (shared across all
#                      cases — the warmup portion is anomaly-free and
#                      identical regardless of anomaly type) and a test-only
#                      file with the group anomalies injected.
#                      Committed to the repo. This IS
#                      anom_types/group.yaml's own default, so no
#                      extra layer is appended for this option.
#   "mini_50pct"    -> the in-repo 377,784-row ERA5 slice
#                      (data/era5/mini_50pct/era5_group_anomalies.csv, ~50% of full-scale).
#                      Committed to the repo. Gives a materially different
#                      (even reversed) architecture ranking than full-scale —
#                      see case00_dataset_scale_impact.ipynb.
#   "mini_28pct"    -> an even smaller in-repo 210,384-row slice
#                      (data/era5/mini_28pct/, ~28% of full-scale). Local-only,
#                      not shipped — see DATA_LICENSE.md.
DATA_SOURCE = "full_split_files"

CASE_ID = "case10_lstm_cyclic_stability"
SESSION_DIR = "runs/regression"
OUTPUT_DIR = f"{SESSION_DIR}/{CASE_ID}"

_data_source_layer = {
    "full_split_files": None,
    "mini_50pct": "../../modules/data_source/mini_50pct_group.yaml",
    "mini_28pct": "../../modules/data_source/mini_28pct_group.yaml",
}[DATA_SOURCE]

_suite_source_path = f"notebooks/cases/{CASE_ID}_suite.yaml"
_suite_dir = os.path.dirname(os.path.abspath(_suite_source_path))

def _resolve(path):
    # base_config / config_layers entries are relative paths meant to be
    # read relative to the suite file's own directory (notebooks/cases/).
    # Resolving them to absolute paths here — rather than leaving them
    # relative — means the resolved copy stays correct no matter how deep
    # under OUTPUT_DIR it ends up being written.
    return path if os.path.isabs(path) else os.path.normpath(os.path.join(_suite_dir, path))

with open(_suite_source_path) as f:
    _suite = yaml.safe_load(f)

_suite["base_config"] = _resolve(_suite["base_config"])
for _run in _suite["runs"]:
    _layers = [_resolve(p) for p in _run["config_layers"]]
    if _data_source_layer:
        _layers.append(_resolve(_data_source_layer))
    _run["config_layers"] = _layers

# Written under OUTPUT_DIR (runs/, already gitignored and read-write) rather
# than notebooks/ (source-controlled, meant to stay read-only) — os.makedirs
# because OUTPUT_DIR won't exist yet on a fresh run.
os.makedirs(OUTPUT_DIR, exist_ok=True)
SUITE_PATH = f"{OUTPUT_DIR}/{CASE_ID}_suite_resolved.yaml"
with open(SUITE_PATH, "w") as f:
    yaml.safe_dump(_suite, f, sort_keys=False)

print(f"DATA_SOURCE = {DATA_SOURCE!r} -> {_data_source_layer}")


## Run the suite

`notebooks/cases/case10_lstm_cyclic_stability_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially. Window-mode runs are batched and fast (well under a minute each on GPU); point-mode runs stream one gradient step per row with no batching, so they run on CPU instead (faster than GPU for this access pattern) and take a few minutes each — `modules/stream/point.yaml` caps them to a 5,000-row subset for this reason.

The suite file itself does not specify a data source — the `DATA_SOURCE` variable (next cell) picks a mode and writes a resolved copy (`{OUTPUT_DIR}/case10_lstm_cyclic_stability_suite_resolved.yaml`) before invoking it:

- `"full_split_files"` (default) — full-scale group-anomaly data, pre-split into a warmup-only file (shared across cases) and a test-only file with the group anomalies injected. Committed to the repo, Colab-portable.
- `"mini_50pct"` — the in-repo 377,784-row ERA5 slice (`data/era5/mini_50pct/era5_group_anomalies.csv`, ~50% of full-scale). Committed to the repo. Gives a materially different architecture ranking — see `case00_dataset_scale_impact.ipynb`.
- `"mini_28pct"` — an even smaller in-repo 210,384-row slice (`data/era5/mini_28pct/`, ~28% of full-scale). Local-only, not shipped — see `DATA_LICENSE.md`.

In [ ]:
!python run_regression.py {SUITE_PATH} \
    --session {SESSION_DIR}

## Compare the runs

`cross_compare.py` walks the session directory, extracts F1/AUC/precision/recall from each run's `trial_predictions.csv`, and writes a performance table plus comparison plots (score histograms, F1-vs-variant line plot, confusion grid, seed-stability heatmap) under `<session>/cross_compare/`.

In [ ]:
!python cross_compare.py {OUTPUT_DIR}

In [ ]:
import pandas as pd
perf = pd.read_csv(f"{OUTPUT_DIR}/cross_compare/performance_table.csv")
perf[["run_name", "arch", "anomaly", "variant", "f1", "auc", "precision", "recall", "tp", "fp", "fn"]]

In [ ]:
# Custom view: F1 over equal-length stream segments (not produced by cross_compare.py).
# This approximates the thesis's 'anomaly-equal bucket' plots (Fig 4.6/4.10/4.11/4.22):
# chop each run's trial_predictions.csv into N equal-length chunks and compute F1 per chunk.
import glob
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

SESSION = OUTPUT_DIR
N_BUCKETS = 8

# leaf dir name (hostname_seed_N) varies by machine, so glob it rather than hardcode it
run_globs = {
    "LSTM, no cyclic": f"{SESSION}/case10_lstm_cyclic_stability/case10_lstm_cyclic_stability_suite/Group/LSTM/cyclic_off/*_seed_42/trial_predictions.csv",
    "LSTM, cyclic input": f"{SESSION}/case10_lstm_cyclic_stability/case10_lstm_cyclic_stability_suite/Group/LSTM/cyclic_on/*_seed_42/trial_predictions.csv",
}
runs = {label: sorted(glob.glob(pattern))[0] for label, pattern in run_globs.items()}

def bucket_f1(csv_path, n_buckets):
    df = pd.read_csv(csv_path, skiprows=1, header=None,
                      names=["ts_start","ts_end","step","confusion","label",
                             "final_pred","trained","final_conf","metrics"])
    df["bucket"] = pd.cut(df["step"], n_buckets, labels=False)
    out = [float("nan")] * n_buckets   # fixed length — a bucket with no rows stays NaN (gap in the line)
    for b, g in df.groupby("bucket"):
        tp = (g["confusion"] == "TP").sum(); fp = (g["confusion"] == "FP").sum()
        fn = (g["confusion"] == "FN").sum()
        prec = tp / (tp + fp) if (tp + fp) else float("nan")
        rec  = tp / (tp + fn) if (tp + fn) else float("nan")
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) and prec == prec and rec == rec and (prec + rec) > 0 else float("nan")
        out[int(b)] = f1
    return out

fig, ax = plt.subplots(figsize=(9, 4))
for label, path in runs.items():
    f1s = bucket_f1(path, N_BUCKETS)
    ax.plot(range(1, N_BUCKETS + 1), f1s, marker="o", label=label)
ax.set_xlabel("Stream segment (1..8, equal length)")
ax.set_ylabel("F1 (within segment)")
ax.set_title("F1 over equal-length stream segments — look for the early dip/recovery pattern")
ax.legend()
ax.set_ylim(-0.05, 1.05)
plt.show()